## tl;dr

Run `python scripts/run_eda_statistical_analysis.py` from the project root to reproduce the executed EDA report, statistical tests, and chart images. This notebook is a companion walkthrough generated from the same project logic.


## Context & Methods

The analysis uses the cleaned CMS analytic table at hospital-condition grain. ERR above 1.0 indicates excess readmissions versus CMS expected performance. Statistical tests use Kruskal-Wallis for categorical comparisons and Spearman correlation for volume versus ERR.


In [ ]:
from pathlib import Path
import pandas as pd
from scipy import stats

ROOT = Path('..').resolve()
df = pd.read_csv(ROOT / 'data/processed/hospital_readmissions_analytic.csv', dtype={'facility_id': 'string', 'zip_code': 'string'}, low_memory=False)
numeric = df[df['excess_readmission_ratio'].notna()].copy()
numeric.shape


## Data

Check source shape and required metric coverage.


In [ ]:
numeric[['facility_id', 'condition', 'excess_readmission_ratio', 'number_of_discharges', 'positive_opportunity_score']].head()


## Results

Condition, state, rating, ownership, and volume tests.


In [ ]:
numeric.groupby('condition').agg(
    rows=('facility_id', 'size'),
    avg_err=('excess_readmission_ratio', 'mean'),
    pct_excess=('excess_readmission_flag', 'mean'),
    opportunity=('positive_opportunity_score', 'sum'),
).sort_values('opportunity', ascending=False)


In [ ]:
groups = [g['excess_readmission_ratio'].to_numpy() for _, g in numeric.dropna(subset=['condition']).groupby('condition')]
stats.kruskal(*groups)


## Takeaways

See `reports/04_eda_statistical_analysis_report.md` for the executed findings and chart outputs.
